# 🎬 WanGP - Vollständiges Colab Notebook (März 2026)

**WanGP** AI Video Generator mit vollständigem Gradio Web Interface, CivitAI LoRA-Support und allen aktuellen Modellen.

## Unterstützte Modelle:
- **Video**: Wan 2.1/2.2, LTX-2/2.3, Hunyuan Video 1.5, SVI2 Pro, LongCat
- **Bild**: Qwen, Z-Image, Flux Klein
- **Audio/TTS**: Qwen3 TTS, Ace Step 1.5, Index TTS 2
- **Editing**: Kiwi Edit, MatAnyone2

## Features:
- ✅ Vollständiges Gradio UI mit allen Plugins
- ✅ **CivitAI LoRA-Integration** - LoRAs direkt herunterladen und verwenden
- ✅ Integrierter CivitAI Browser & Downloader
- ✅ Unterstützung für quantisierte Modelle (int8, fp8, GGUF)
- ✅ Accelerator LoRAs (CausVid, AccVid, FusioniX) für schnellere Generation
- ✅ Google Drive für persistente Speicherung
- ✅ Performance-Optimierungen für Colab

## Voraussetzungen:
- GPU Runtime (T4 minimum, L4/A100 empfohlen)
- High RAM (falls verfügbar)

---

**Anleitung:** Führe die Zellen der Reihe nach aus (▶️ oder Shift+Enter)

## 📋 Schritt 1: GPU Check & Runtime Setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
print(f"\n{'='*60}")
print("SYSTEM INFORMATIONEN")
print(f"{'='*60}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfuegbar: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM Total: {total_vram:.2f} GB")

    print(f"\n{'='*60}")
    if total_vram >= 40:
        print("A100 erkannt - Alle Modelle und hohe Aufloesungen moeglich")
        print("Empfehlung: LTX-2 Dev 19B, Wan 2.2 14B, 1080p Videos")
    elif total_vram >= 24:
        print("L4/A10 erkannt - Die meisten Modelle funktionieren gut")
        print("Empfehlung: LTX-2 Distilled, Wan 2.2 14B, 720p Videos")
    elif total_vram >= 15:
        print("T4 erkannt - Basis Modelle mit niedrigeren Aufloesungen")
        print("Empfehlung: LTX-2 Distilled FP8, Wan 2.2 5B, 480p Videos")
    else:
        print("Wenig VRAM - Upgrade zu einer besseren GPU empfohlen")
    print(f"{'='*60}")
else:
    print("\nFEHLER: Keine GPU gefunden!")
    print("Bitte Runtime aendern: Runtime -> Change runtime type -> GPU")

NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB

SYSTEM INFORMATIONEN
PyTorch Version: 2.10.0+cu128
CUDA verfuegbar: True
CUDA Version: 12.8
GPU: NVIDIA A100-SXM4-40GB
VRAM Total: 39.49 GB

L4/A10 erkannt - Die meisten Modelle funktionieren gut
Empfehlung: LTX-2 Distilled, Wan 2.2 14B, 720p Videos


## 📦 Schritt 2: Google Drive Mount (Optional)

Empfohlen um Modelle und LoRAs persistent zu speichern.

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

wan_drive = '/content/drive/MyDrive/WanGP'
os.makedirs(f'{wan_drive}/models', exist_ok=True)
os.makedirs(f'{wan_drive}/output', exist_ok=True)
os.makedirs(f'{wan_drive}/loras', exist_ok=True)

print(f"Drive gemountet: {wan_drive}")
print(f"  Modelle:  {wan_drive}/models")
print(f"  Output:   {wan_drive}/output")
print(f"  LoRAs:    {wan_drive}/loras")

Mounted at /content/drive
Drive gemountet: /content/drive/MyDrive/WanGP
  Modelle:  /content/drive/MyDrive/WanGP/models
  Output:   /content/drive/MyDrive/WanGP/output
  LoRAs:    /content/drive/MyDrive/WanGP/loras


## 🔧 Schritt 3: WanGP Installation

Klont das aktuelle Repository und installiert alle Abhaengigkeiten.

In [2]:
import os

if not os.path.exists('/content/Wan2GP'):
    print("Clone WanGP Repository...")
    !git clone https://github.com/deepbeepmeep/Wan2GP.git /content/Wan2GP
    print("Repository geklont")
else:
    print("Repository existiert bereits, aktualisiere...")
    !cd /content/Wan2GP && git pull

%cd /content/Wan2GP

# System Dependencies
print("\nInstalliere System Dependencies...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsm6 libxext6 libxrender-dev > /dev/null 2>&1
print("System Dependencies installiert")

Clone WanGP Repository...
Cloning into '/content/Wan2GP'...
remote: Enumerating objects: 9967, done.
remote: Counting objects: 100% (396/396), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 9967 (delta 263), reused 236 (delta 216), pack-reused 9571 (from 3)
Receiving objects: 100% (9967/9967), 34.43 MiB | 15.30 MiB/s, done.
Resolving deltas: 100% (6251/6251), done.
Repository geklont
/content/Wan2GP

Installiere System Dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
System Dependencies installiert


In [3]:
print("Installiere Python Pakete (kann 5-10 Minuten dauern)...\n")

!pip install -q -U pip setuptools wheel

# PyTorch passend zur Colab CUDA Version
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# WanGP Requirements
!pip install -q -r requirements.txt 2>&1 | tail -5

# Zusaetzliche Pakete fuer Colab und CivitAI
!pip install -q pyngrok requests tqdm

print("\nAlle Pakete installiert!")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")

Installiere Python Pakete (kann 5-10 Minuten dauern)...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
mcp 1.26.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompat

## 🔗 Schritt 4: Google Drive Verknuepfung

Verknuepft LoRA- und Model-Ordner mit Google Drive fuer persistente Speicherung.

In [5]:
import os

wan_drive = '/content/drive/MyDrive/WanGP'

# LoRA-Verzeichnisse mit Drive verknuepfen
if os.path.exists(wan_drive):
    lora_base = '/content/Wan2GP/loras'
    os.makedirs(lora_base, exist_ok=True)

    # LoRA-Unterordner fuer verschiedene Modelle
    lora_dirs = ['wan', 'wan_5B', 'wan_1.3B', 'wan_i2v', 'hunyuan',
                 'hunyuan_i2v', 'ltxv', 'flux', 'flux2', 'qwen', 'z_image']

    for d in lora_dirs:
        drive_lora = f'{wan_drive}/loras/{d}'
        local_lora = f'{lora_base}/{d}'
        os.makedirs(drive_lora, exist_ok=True)
        if not os.path.exists(local_lora):
            os.symlink(drive_lora, local_lora)

    print("LoRA-Verzeichnisse mit Google Drive verknuepft:")
    for d in lora_dirs:
        count = len([f for f in os.listdir(f'{wan_drive}/loras/{d}')
                     if f.endswith('.safetensors')])
        if count > 0:
            print(f"  loras/{d}: {count} LoRA(s) gefunden")
        else:
            print(f"  loras/{d}: (leer)")

    # Output mit Drive verknuepfen
    output_dir = '/content/Wan2GP/output'
    if not os.path.exists(output_dir):
        os.symlink(f'{wan_drive}/output', output_dir)
        print(f"\nOutput-Verzeichnis: {wan_drive}/output")
else:
    print("Kein Google Drive - LoRAs und Output werden lokal gespeichert")
    os.makedirs('/content/Wan2GP/loras', exist_ok=True)

LoRA-Verzeichnisse mit Google Drive verknuepft:
  loras/wan: (leer)
  loras/wan_5B: (leer)
  loras/wan_1.3B: (leer)
  loras/wan_i2v: (leer)
  loras/hunyuan: (leer)
  loras/hunyuan_i2v: (leer)
  loras/ltxv: (leer)
  loras/flux: (leer)
  loras/flux2: (leer)
  loras/qwen: (leer)
  loras/z_image: (leer)

Output-Verzeichnis: /content/drive/MyDrive/WanGP/output


## 🎨 Schritt 5: CivitAI LoRA Download

Hier kannst du LoRAs von CivitAI herunterladen und direkt verwenden.

### So findest du LoRAs:
1. Gehe auf [civitai.com](https://civitai.com)
2. Filtere nach **LoRA** und suche z.B. nach "Wan", "Wan 2.1", "Hunyuan Video", "LTX"
3. Kopiere die **Model-Version-ID** aus der URL (die Zahl am Ende)
4. Oder kopiere den **kompletten Download-Link**

### CivitAI API Key (optional, aber empfohlen):
Einige Modelle erfordern einen API Key. Du kannst ihn unter [civitai.com/user/account](https://civitai.com/user/account) -> API Keys erstellen.

In [6]:
import os
import requests
from tqdm.notebook import tqdm
import getpass
import re
import json
from pathlib import Path

# ======================================
# CivitAI API Key (optional)
# ======================================
print("CivitAI API Key (optional, fuer geschuetzte Modelle):")
print("Einfach Enter druecken um zu ueberspringen.\n")
CIVITAI_API_KEY = getpass.getpass("CivitAI API Key: ").strip()

if CIVITAI_API_KEY:
    print("API Key gesetzt")
else:
    print("Kein API Key - nur oeffentliche Modelle verfuegbar")


def get_civitai_model_info(model_version_id):
    """Holt Informationen zu einer CivitAI Model-Version."""
    url = f"https://civitai.com/api/v1/model-versions/{model_version_id}"
    headers = {}
    if CIVITAI_API_KEY:
        headers["Authorization"] = f"Bearer {CIVITAI_API_KEY}"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()


def download_civitai_lora(model_version_id, target_model="wan", filename=None):
    """
    Laedt ein LoRA von CivitAI herunter.

    Args:
        model_version_id: Die Model-Version-ID von CivitAI (Zahl aus der URL)
        target_model: Zielordner - einer von:
            'wan'       - Wan 2.1/2.2 Text-to-Video (14B)
            'wan_5B'    - Wan 5B Variante
            'wan_1.3B'  - Wan 1.3B Variante
            'wan_i2v'   - Wan Image-to-Video
            'hunyuan'   - Hunyuan Video
            'hunyuan_i2v' - Hunyuan Image-to-Video
            'ltxv'      - LTX Video / LTX-2
            'flux'      - Flux
            'flux2'     - Flux 2
            'qwen'      - Qwen
            'z_image'   - Z-Image
        filename: Optionaler Dateiname (Standard: Original-Name von CivitAI)
    """
    lora_dir = f'/content/Wan2GP/loras/{target_model}'
    os.makedirs(lora_dir, exist_ok=True)

    # Model-Info abrufen
    print(f"Lade Model-Info fuer Version {model_version_id}...")
    try:
        info = get_civitai_model_info(model_version_id)
        model_name = info.get('model', {}).get('name', 'Unbekannt')
        version_name = info.get('name', '')
        print(f"Modell: {model_name} - {version_name}")

        # Finde die safetensors-Datei
        dl_file = None
        for f in info.get('files', []):
            if f['name'].endswith('.safetensors'):
                dl_file = f
                break
        if not dl_file and info.get('files'):
            dl_file = info['files'][0]

        if not dl_file:
            print("FEHLER: Keine herunterladbare Datei gefunden")
            return None

        if not filename:
            filename = dl_file['name']
        file_size = dl_file.get('sizeKB', 0) * 1024

        # Trigger Words anzeigen
        trigger_words = info.get('trainedWords', [])
        if trigger_words:
            print(f"Trigger Words: {', '.join(trigger_words)}")

    except Exception as e:
        print(f"Warnung: Konnte Model-Info nicht laden ({e})")
        print("Versuche direkten Download...")
        if not filename:
            filename = f"civitai_{model_version_id}.safetensors"
        file_size = 0

    filepath = os.path.join(lora_dir, filename)

    # Pruefen ob bereits vorhanden
    if os.path.exists(filepath):
        print(f"Bereits vorhanden: {filepath}")
        return filepath

    # Download
    download_url = f"https://civitai.com/api/download/models/{model_version_id}"
    headers = {}
    if CIVITAI_API_KEY:
        headers["Authorization"] = f"Bearer {CIVITAI_API_KEY}"

    print(f"Lade herunter: {filename}")
    if file_size > 0:
        print(f"Groesse: {file_size / 1024 / 1024:.1f} MB")

    resp = requests.get(download_url, headers=headers, stream=True)
    resp.raise_for_status()

    total = int(resp.headers.get('content-length', file_size)) or None
    with open(filepath, 'wb') as f:
        with tqdm(total=total, unit='B', unit_scale=True, desc=filename) as pbar:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))

    print(f"Gespeichert: {filepath}")
    return filepath


def download_civitai_lora_by_url(url, target_model="wan"):
    """
    Laedt ein LoRA anhand einer CivitAI URL herunter.

    Akzeptiert URLs wie:
        https://civitai.com/models/12345/model-name
        https://civitai.com/models/12345?modelVersionId=67890
        https://civitai.com/api/download/models/67890
    """
    # Extrahiere Model-Version-ID aus verschiedenen URL-Formaten
    version_match = re.search(r'modelVersionId=(\d+)', url)
    if version_match:
        return download_civitai_lora(version_match.group(1), target_model)

    dl_match = re.search(r'/api/download/models/(\d+)', url)
    if dl_match:
        return download_civitai_lora(dl_match.group(1), target_model)

    # Model-Seite -> hole neueste Version
    model_match = re.search(r'/models/(\d+)', url)
    if model_match:
        model_id = model_match.group(1)
        print(f"Lade Model-Info fuer Model {model_id}...")
        api_url = f"https://civitai.com/api/v1/models/{model_id}"
        headers = {}
        if CIVITAI_API_KEY:
            headers["Authorization"] = f"Bearer {CIVITAI_API_KEY}"
        resp = requests.get(api_url, headers=headers)
        resp.raise_for_status()
        data = resp.json()
        versions = data.get('modelVersions', [])
        if versions:
            version_id = versions[0]['id']
            print(f"Neueste Version: {versions[0].get('name', version_id)}")
            return download_civitai_lora(version_id, target_model)

    print(f"FEHLER: Konnte keine Model-Version-ID aus URL extrahieren: {url}")
    return None


def search_civitai_loras(query, model_type="LORA", limit=10):
    """
    Sucht nach LoRAs auf CivitAI.

    Args:
        query: Suchbegriff (z.B. 'wan anime', 'hunyuan video style')
        model_type: Typ (Standard: LORA)
        limit: Max Ergebnisse
    """
    url = "https://civitai.com/api/v1/models"
    params = {
        'query': query,
        'types': model_type,
        'limit': limit,
        'sort': 'Most Downloaded',
    }
    headers = {}
    if CIVITAI_API_KEY:
        headers["Authorization"] = f"Bearer {CIVITAI_API_KEY}"

    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    data = resp.json()

    print(f"\nSuchergebnisse fuer '{query}':")
    print(f"{'='*70}")
    results = data.get('items', [])
    for i, item in enumerate(results, 1):
        name = item.get('name', 'Unbekannt')
        downloads = item.get('stats', {}).get('downloadCount', 0)
        rating = item.get('stats', {}).get('rating', 0)
        model_id = item.get('id')
        versions = item.get('modelVersions', [])
        version_id = versions[0]['id'] if versions else '?'
        tags = ', '.join(item.get('tags', [])[:3])

        print(f"\n{i}. {name}")
        print(f"   Downloads: {downloads:,} | Rating: {rating:.1f}")
        print(f"   Tags: {tags}")
        print(f"   Version-ID: {version_id}")
        print(f"   URL: https://civitai.com/models/{model_id}")

    print(f"\n{'='*70}")
    print(f"\nZum Download: download_civitai_lora(VERSION_ID, 'wan')")
    return results


print("CivitAI LoRA-Funktionen geladen:")
print("  search_civitai_loras('wan anime')     - LoRAs suchen")
print("  download_civitai_lora(VERSION_ID, 'wan') - LoRA herunterladen")
print("  download_civitai_lora_by_url(URL, 'wan') - LoRA per URL laden")

CivitAI API Key (optional, fuer geschuetzte Modelle):
Einfach Enter druecken um zu ueberspringen.

CivitAI API Key: ··········
API Key gesetzt
CivitAI LoRA-Funktionen geladen:
  search_civitai_loras('wan anime')     - LoRAs suchen
  download_civitai_lora(VERSION_ID, 'wan') - LoRA herunterladen
  download_civitai_lora_by_url(URL, 'wan') - LoRA per URL laden


### 🔍 CivitAI LoRAs suchen

Passe den Suchbegriff an um passende LoRAs zu finden.

In [8]:
# ======================================
# CivitAI LoRA Suche
# Aendere den Suchbegriff nach Bedarf!
# ======================================

# Beispiele:
# search_civitai_loras('wan 2.1 anime')
# search_civitai_loras('wan video style')
# search_civitai_loras('hunyuan video')
# search_civitai_loras('ltx video lora')

results = search_civitai_loras('ltx video')


Suchergebnisse fuer 'ltx video':

1. LTX-2.3 T2V Police Rabbit of Judy Hopps LoRA
   Downloads: 158 | Rating: 0.0
   Tags: character, girl, zootopia
   Version-ID: 2757326
   URL: https://civitai.com/models/2452203


Zum Download: download_civitai_lora(VERSION_ID, 'wan')


### 📥 CivitAI LoRAs herunterladen

Lade LoRAs herunter indem du die Version-ID oder die URL angibst.

**Wichtig:** Waehle den richtigen `target_model` Ordner:
- `'wan'` - Wan 2.1/2.2 Text-to-Video (14B)
- `'wan_5B'` - Wan 5B
- `'wan_i2v'` - Wan Image-to-Video
- `'ltxv'` - LTX Video / LTX-2
- `'hunyuan'` - Hunyuan Video
- `'flux'` - Flux

In [11]:
# ======================================
# LoRA Download - Passe die Werte an!
# ======================================

# Option 1: Download per Version-ID (aus der Suche oben)
# download_civitai_lora(VERSION_ID, target_model='wan')

# Option 2: Download per CivitAI URL
# download_civitai_lora_by_url('https://civitai.com/models/XXXXX/model-name', target_model='wan')

# Option 3: Mehrere LoRAs auf einmal
# loras_to_download = [
#     {'version_id': 123456, 'target': 'wan'},
#     {'version_id': 789012, 'target': 'ltxv'},
# ]
# for lora in loras_to_download:
#     download_civitai_lora(lora['version_id'], lora['target'])

print("Entkommentiere die gewuenschte Option oben und fuehre die Zelle aus.")
print("\nBeispiel:")
print("  download_civitai_lora(123456, 'wan')")
print("  download_civitai_lora_by_url('https://civitai.com/models/...', 'wan')")

Entkommentiere die gewuenschte Option oben und fuehre die Zelle aus.

Beispiel:
  download_civitai_lora(123456, 'wan')
  download_civitai_lora_by_url('https://civitai.com/models/...', 'wan')


In [12]:
# LoRA-Uebersicht: Zeigt alle heruntergeladenen LoRAs
import os

lora_base = '/content/Wan2GP/loras'
if os.path.exists(lora_base):
    print("Installierte LoRAs:")
    print(f"{'='*60}")
    total = 0
    for d in sorted(os.listdir(lora_base)):
        dp = os.path.join(lora_base, d)
        if os.path.isdir(dp):
            files = [f for f in os.listdir(dp) if f.endswith(('.safetensors', '.pt', '.bin'))]
            if files:
                print(f"\n  {d}/")
                for f in files:
                    size_mb = os.path.getsize(os.path.join(dp, f)) / 1024 / 1024
                    print(f"    {f} ({size_mb:.1f} MB)")
                    total += 1
    if total == 0:
        print("  (keine LoRAs gefunden)")
    else:
        print(f"\nGesamt: {total} LoRA(s)")
else:
    print("LoRA-Verzeichnis existiert noch nicht.")

Installierte LoRAs:

  wan/
    NSFW-22-L-e8.safetensors (585.1 MB)
    NSFW-22-H-e8.safetensors (585.1 MB)

Gesamt: 2 LoRA(s)


## 🔑 Schritt 6: Hugging Face Login (Optional)

Fuer Modelle die einen HF Account erfordern.

In [ ]:
import getpass

print("Hugging Face Token (optional):")
print("Fuer die meisten Modelle nicht noetig. Enter zum Ueberspringen.\n")

hf_token = getpass.getpass("HF Token: ").strip()
if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token)
        print("Bei Hugging Face eingeloggt")
    except Exception as e:
        print(f"Login fehlgeschlagen: {e}")
else:
    print("Uebersprungen")

## 🚀 Schritt 7: WanGP Starten

Startet das vollstaendige Gradio Interface mit oeffentlicher URL.

**Nach dem Start:**
1. Kopiere die `gradio.live` URL aus dem Output
2. Oeffne sie im Browser
3. Nutze das vollstaendige WanGP Interface mit LoRA-Support!

**LoRAs verwenden:**
- Im UI: Gehe zu **Advanced Tab** -> **Loras**
- Waehle die heruntergeladenen LoRAs aus dem Dropdown
- Setze den Multiplikator (Standard: 1.0)
- Der integrierte **CivitAI Browser** ist ebenfalls im UI verfuegbar

In [13]:
import os
os.chdir('/content/Wan2GP')

# Performance-Einstellungen
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

print("Starte WanGP mit Gradio UI...")
print("="*70)
print("Modelle werden beim ersten Start automatisch heruntergeladen.")
print("LoRAs sind unter Advanced -> Loras im UI verfuegbar.")
print("Der CivitAI Browser ist als Plugin integriert.")
print("="*70)
print("\nBitte warten...\n")

!python wgp.py --share --server-name 0.0.0.0

Starte WanGP mit Gradio UI...
Modelle werden beim ersten Start automatisch heruntergeladen.
LoRAs sind unter Advanced -> Loras im UI verfuegbar.
Der CivitAI Browser ist als Plugin integriert.

Bitte warten...

2026-03-12 19:29:32.521737: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 19:29:32.541195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773343772.563603   14571 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773343772.571200   14571 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to regis

## 📊 Schritt 8: Monitoring & Tools

In [ ]:
# GPU Monitoring
!nvidia-smi

import torch
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\nPyTorch Memory: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")

In [ ]:
# Memory Cleanup
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    alloc = torch.cuda.memory_allocated() / 1024**3
    print(f"Cleanup abgeschlossen. {alloc:.2f} GB belegt.")

## 💡 Anleitung: LoRAs in WanGP verwenden

### 1. LoRA in den richtigen Ordner legen
Die LoRA `.safetensors` Datei muss im passenden Unterordner von `loras/` liegen:

| Modell | LoRA-Ordner |
|--------|------------|
| Wan 2.1/2.2 (14B) | `loras/wan/` |
| Wan 5B | `loras/wan_5B/` |
| Wan 1.3B | `loras/wan_1.3B/` |
| Wan Image-to-Video | `loras/wan_i2v/` |
| Hunyuan Video | `loras/hunyuan/` |
| LTX Video / LTX-2 | `loras/ltxv/` |
| Flux | `loras/flux/` |
| Qwen | `loras/qwen/` |

### 2. Im UI aktivieren
- Oeffne das **Advanced** Tab
- Gehe zum **Loras** Bereich
- Waehle die LoRA aus dem Dropdown
- Setze den **Multiplikator** (empfohlen: 0.7-1.2)

### 3. Multiplikator-Syntax
- `1.0` - Einfacher Multiplikator
- `0.9,0.8,0.7` - Zeitbasiert (staerker am Anfang)
- `0;1` - Phasenbasiert fuer Wan 2.2 (High Noise; Low Noise)
- `1;0.5` - LTX-2 Multi-Pass (Pass 1; Pass 2)

### 4. Accelerator LoRAs (schnellere Generation)
- **CausVid**: 4-12 Steps, Guidance=1, Shift=7
- **AccVid**: Gleiche Steps, kein CFG noetig
- **FusioniX**: Accelerator + Style, Guidance=1, Shift=2
- **Lightx2v 4-steps**: Fuer Wan 2.2, Multiplikator `1;0 0;1`

### 5. Integrierter CivitAI Browser
WanGP hat einen eingebauten CivitAI Browser als Plugin. Du kannst direkt aus dem UI heraus LoRAs suchen und herunterladen.

---

## GPU Empfehlungen

### T4 (15 GB VRAM):
- LTX-2 Distilled FP8: 720p, 10s Videos
- Wan 2.2 5B: 480p Videos
- 8 Steps (distilled) oder 20 Steps

### L4/A10 (24 GB VRAM):
- LTX-2 Distilled: 720-1080p, 10s Videos
- Wan 2.2 14B: 720p Videos
- Alle Modelle funktionieren

### A100 (40+ GB VRAM):
- LTX-2 Dev 19B: 1080p, 20s Videos mit Audio
- Wan 2.2 14B: 1080p Videos
- Hohe Quality Settings, alle LoRAs

---

## Troubleshooting

**"CUDA Out of Memory"**: Memory Cleanup Cell ausfuehren, Aufloesung/Frames reduzieren

**"No module named ..."**: Dependency Cell nochmal ausfuehren

**Gradio URL laedt nicht**: 1-2 Min warten, URL aus Output kopieren

**LoRA wird nicht angezeigt**: Im UI auf "Refresh" klicken, pruefen ob Datei im richtigen Ordner liegt

**CivitAI Download schlaegt fehl**: API Key setzen, URL/Version-ID pruefen

---

**Links:**
- [WanGP GitHub](https://github.com/deepbeepmeep/Wan2GP)
- [WanGP LoRA Docs](https://github.com/deepbeepmeep/Wan2GP/blob/main/docs/LORAS.md)
- [CivitAI LoRAs](https://civitai.com/models?types=LORA)
- [WanGP Changelog](https://github.com/deepbeepmeep/Wan2GP/blob/main/docs/CHANGELOG.md)